In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:
df = pd.read_csv("../data/bedtime_screentime_sleep_debt.csv")

numerical_features = [
    "total_sleep_hours",
    "morning_alarm_snoozes",
    "bedtime_phone_minutes",
    "sleep_latency_min",
    "blue_light_filter_active",
    "screen_brightness_pct",
    "caffeine_post_5pm_mg",
    "deep_sleep_pct"
]

categorical_features = [
    "chronotype",
    "occupation_type",
    "primary_bedtime_app"
]

target = "sleep_debt_category"

X = df[numerical_features + categorical_features]
y = df[target]

SPLIT TRAIN AND TEST

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

PIPELINES

In [4]:
numerical_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

COMBINED PREPROCESSOR

In [5]:
preprocessor = ColumnTransformer([
    (
        "numerical",
        numerical_pipeline,
        numerical_features
    ),

    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])

INSTANTIATING CROSS-VALIDATOR, MODELS, AND EVALUATION

In [6]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

models = {

    "Logistic Regression": (
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        ),

        {
            "model__C": [0.1, 1, 10]
        }
    ),

    "Random Forest": (
        RandomForestClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),

        {
            "model__n_estimators": [150],
            "model__max_depth": [None, 15]
        }
    ),

    "Decision Tree": (
        DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42
        ),

        {
            "model__max_depth": [None, 5, 10, 15],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4]
        }
    )
}

results = []
all_results = []
best_models = {}

scoring = {
    "f1_macro": "f1_macro",
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro"
}

for name, (model, parameters) in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    grid = GridSearchCV(
        pipeline,
        parameters,
        scoring=scoring,
        cv=cv,
        n_jobs=-1,
        refit="f1_macro"
    )

    grid.fit(X_train, y_train)

    best_models[name] = grid

    results.append({
        "Model": name,
        "Best CV Macro-F1": grid.best_score_,
        "Best Parameters": grid.best_params_
    })

    all_results.append({
      "Model": name,
      "Best CV Macro-F1": grid.cv_results_["mean_test_f1_macro"][grid.best_index_],
      "CV Accuracy": grid.cv_results_["mean_test_accuracy"][grid.best_index_],
      "CV Precision Macro": grid.cv_results_["mean_test_precision_macro"][grid.best_index_],
      "CV Recall Macro": grid.cv_results_["mean_test_recall_macro"][grid.best_index_],
      "Best Parameters": grid.best_params_
  })

results_df = pd.DataFrame(results)
all_results_df = pd.DataFrame(all_results)


PRINTING THE EVALUATION

In [7]:
print(results_df)
print(results_df.loc[1, 'Best Parameters'])


print("\n" + "=" * 150)
print("ALL GRID SEARCH RESULTS")
print("=" * 150)

print(
    all_results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

                 Model  Best CV Macro-F1  \
0  Logistic Regression          0.939492   
1        Random Forest          0.955369   
2        Decision Tree          0.943784   

                                     Best Parameters  
0                                   {'model__C': 10}  
1  {'model__max_depth': None, 'model__n_estimator...  
2  {'model__max_depth': 10, 'model__min_samples_l...  
{'model__max_depth': None, 'model__n_estimators': 150}

ALL GRID SEARCH RESULTS
              Model  Best CV Macro-F1  CV Accuracy  CV Precision Macro  CV Recall Macro                                                                       Best Parameters
Logistic Regression            0.9395       0.9581              0.9176           0.9698                                                                      {'model__C': 10}
      Random Forest            0.9554       0.9747              0.9459           0.9664                                {'model__max_depth': None, 'model__n_estimators': 150}
 

In [8]:
import joblib

# Get the best Random Forest pipeline
best_model = best_models["Random Forest"].best_estimator_

# Save the complete pipeline
joblib.dump(
    best_model,
    "../models/sleep_debt_model.pkl"
)

print("Best model saved successfully.")

Best model saved successfully.
